** Calculate Correlation Coefficient between station observations and incident pixels **


In [ ]:
import xarray
import pandas as pd
import glob
import os
import rasterio
from scipy.stats import spearmanr

In [4]:
#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1990_2019.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])

#Read gridded data - starting with era5L
era5L_path = r"\\192.168.1.210\giis_home\Projects\2025\2025_ClimateDataCorrection\Datasets\era5LandDaily\precipitation"
era5L_tiff_files = glob.glob(era5L_path + "/*.tif")
#era5L_tiff_files

#helper function - extract date from filename
def extract_date(file):
    filename = os.path.basename(file)
    return filename[11:19]

era5L_dates = list(map(extract_date, era5L_tiff_files))
#era5L_dates

FileNotFoundError: [Errno 2] No such file or directory: '..\\Data\\Processed\\Insitu\\insitu_rainfall_1990_2019.csv'

In [34]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
unique_stn   

,index_no,lat,long
0,102,29.55744,80.41317
10804,103,29.46714,80.54575
21761,104,29.30135,80.58775
32718,105,28.95477,80.23049
43207,107,29.84496,80.54539
...,...,...,...
852821,1407,26.90964,87.92308
862846,1408,26.67067,87.70318
873709,1412,26.56389,88.06150
884545,1416,26.86806,88.07833


In [36]:
#extract values from tiff files using rasterio
prec_records = []
for file in era5L_tiff_files[:10]:
    date_obj = extract_date(file)  
    try:
        with rasterio.open(file) as src:
            crs = src.crs
            if crs == "EPSG:4326":
                sampled_values = list(src.sample(stn_points))
                
                for i, value in enumerate(sampled_values):
                    prec_records.append({
                        'date': date_obj,
                        'stn_id': unique_stn.iloc[i]['index_no'],
                        'prec_grid': value[0]
                        })
                 
    except Exception as e:
        print(e)

df_grid = pd.DataFrame(prec_records)
df_grid

,date,stn_id,prec_grid
0,20150217,102.0,0.153902
1,20150217,103.0,0.298607
2,20150217,104.0,0.550869
3,20150217,105.0,1.433167
4,20150217,107.0,1.133758
...,...,...,...
855,20100712,1407.0,44.870928
856,20100712,1408.0,30.361891
857,20100712,1412.0,59.032306
858,20100712,1416.0,65.206394


In [37]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid['date'] = pd.to_datetime(df_grid['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid, df_insitu1, on=['date','stn_id'], how='inner')

#correlation coefficient
correlation = spearmanr(df_merged['prec_grid'], df_merged['prec_insitu'], nan_policy='omit') 
print("Correlation (Spearman) Value: ",correlation.correlation)  
print("p-value:", correlation.pvalue)


Correlation (Spearman) Value:  0.544266466617433
p-value: 8.788056178957644e-52
